Efficiency

Load BESStie

In [ ]:
# a lot of ignore warnings were creating noisy outputs, so filter these away.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
test_dataframe = dataset['test'].to_pandas()

Logistic Regression, SVM, RoBERTa and DeBERTa comparison

In [ ]:
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
roberta_model.eval()

model_name = "microsoft/deberta-v3-base"
deberta_tokenizer = AutoTokenizer.from_pretrained(model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, torch_dtype=torch.float32).to(device)
deberta_model.eval()

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])

lr_sarcasm = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])

svm_sarcasm = LinearSVC(max_iter=10000, class_weight='balanced')
svm_sarcasm.fit(X_train, training_dataframe['Sarcasm'])

def transformer_timings(model, tokenizer, texts, device, n_runs=50):
    times = []
    for _ in range(n_runs):
        start = time.time()
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elif device.type == "mps":
            torch.mps.synchronize()
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)

def sklearn_timings(model, vectorizer, texts, n_runs=50):
    times = []
    for _ in range(n_runs):
        start = time.time()
        X = vectorizer.transform(texts)
        model.predict(X)
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)

def warmup(model, tokenizer, texts, device):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        model(**inputs)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

test_texts = test_dataframe['text'].tolist()
batch_sizes = [1, 16, 64, 256, 1024]

# WARMUP
warmup(roberta_model, roberta_tokenizer, test_texts[:1], device)
warmup(deberta_model, deberta_tokenizer, test_texts[:1], device)
vectorizer.transform(test_texts[:1])
lr_sarcasm.predict(vectorizer.transform(test_texts[:1]))
svm_sarcasm.predict(vectorizer.transform(test_texts[:1]))

# TIMING
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Apple MPS' if torch.backends.mps.is_available() else 'CPU'}\n")

for size in batch_sizes:
    texts = test_texts[:size]

    lr_mean, lr_std = sklearn_timings(lr_sarcasm, vectorizer, texts)
    svm_mean, svm_std = sklearn_timings(svm_sarcasm, vectorizer, texts)
    rob_mean, rob_std = transformer_timings(roberta_model, roberta_tokenizer, texts, device)
    deb_mean, deb_std = transformer_timings(deberta_model, deberta_tokenizer, texts, device)

    rob_ratio = rob_mean / lr_mean
    deb_ratio = deb_mean / lr_mean

    print(f"Batch {size:>4} | LR: {lr_mean:>8.2f} ± {lr_std:>6.2f} ms | SVM: {svm_mean:>8.2f} ± {svm_std:>6.2f} ms | RoBERTa: {rob_mean:>8.2f} ± {rob_std:>6.2f} ms ({rob_ratio:>6.1f}x) | DeBERTa: {deb_mean:>8.2f} ± {deb_std:>6.2f} ms ({deb_ratio:>6.1f}x)")

Compare SVM vs Logistic Regression inference timings

In [ ]:
lr_sentiment = LogisticRegression(max_iter=1000)
lr_sentiment.fit(X_train, training_dataframe['Sentiment'])

# Warmup
vectorizer.transform(test_texts[:1])
lr_sentiment.predict(vectorizer.transform(test_texts[:1]))

X_train = vectorizer.fit_transform(training_dataframe['text'])
svm_sentiment = LinearSVC(max_iter=10000, class_weight='balanced')
svm_sentiment.fit(X_train, training_dataframe['Sarcasm'])
svm_sentiment.predict(vectorizer.transform(test_texts[:1]))

for label, model in [("LR", lr_sentiment), ("SVM", svm_sentiment)]:
    times = []
    for _ in range(50):
        start = time.time()
        X = vectorizer.transform(test_texts[:1])
        model.predict(X)
        times.append((time.time() - start) * 1000)
    print(f"{label}: {np.mean(times):.3f} ± {np.std(times):.3f} ms")

Load the LoRA model

In [9]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel


def load(base_id, adapter_repo):
    tok = AutoTokenizer.from_pretrained(base_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForSequenceClassification.from_pretrained(
        base_id, num_labels=2, dtype=torch.bfloat16,
    )
    base.config.pad_token_id = tok.pad_token_id
    model = PeftModel.from_pretrained(base, adapter_repo)
    model.eval()
    if torch.cuda.is_available():
        model.to('cuda')
    return tok, model


@torch.no_grad()
def predict(tok, model, text, max_len=256):
    inputs = tok(text, return_tensors='pt', truncation=True, max_length=max_len, padding=True)
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    logits = model(**inputs).logits
    return int(logits.argmax(dim=-1))


if __name__ == "__main__":
    tok, model = load(
        "Qwen/Qwen2.5-1.5B",
        "berkinksk/besstie-sarcasm-en-UK-Qwen2.5-1.5B-seed2024",
    )

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Timings for LoRA models

In [ ]:
import time
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def load(base_id, adapter_repo):
    tok = AutoTokenizer.from_pretrained(base_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForSequenceClassification.from_pretrained(
        base_id, num_labels=2, torch_dtype=torch.bfloat16,
    )
    base.config.pad_token_id = tok.pad_token_id
    model = PeftModel.from_pretrained(base, adapter_repo)
    model.eval()
    model.to(device)
    return tok, model

def lora_timings(tok, model, texts, n_runs=50):
    times = []
    for _ in range(n_runs):
        start = time.time()
        inputs = tok(texts, return_tensors='pt', truncation=True, max_length=256, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            model(**inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elif device.type == "mps":
            torch.mps.synchronize()
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)

test_texts = test_dataframe['text'].tolist()

lora_configs = [
    {"Qwen/Qwen2.5-1.5B", "berkinksk/besstie-sarcasm-en-UK-Qwen2.5-1.5B-seed2024", "LoRA Qwen (UK)"},
    ("allenai/OLMo-2-0425-1B", "berkinksk/besstie-sarcasm-en-AU-OLMo-2-0425-1B-seed2024", "LoRA OLMo (AU)"),
    ("Qwen/Qwen2.5-1.5B", "berkinksk/besstie-sarcasm-en-IN-Qwen2.5-1.5B-seed2024", "LoRA Qwen (IN)"),
]

for base_id, adapter_repo, label in lora_configs:
    print(f"\nLoading {label}...")
    tok, model = load(base_id, adapter_repo)
    
    # Warmup
    inputs = tok(test_texts[:1], return_tensors='pt', truncation=True, max_length=256, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        model(**inputs)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()
    
    # Time across batch sizes
    for size in [1, 16, 64, 256]:
        texts = test_texts[:size]
        mean_ms, std_ms = lora_timings(tok, model, texts)
        print(f"  Batch {size:>4}: {mean_ms:>8.2f} ± {std_ms:>6.2f} ms")